# The Urn Draw EV Engine

A legendary quant interview question format (common at SIG and Jane Street) involves an Urn containing Red and Black balls without replacement. 

**The Setup:**
- You have `R` red balls and `B` black balls in an urn.
- At any point, you can `STOP` and take a guaranteed payout (often $0, or sometimes a positive number like $10).
- If you `DRAW`:
  - Drawing a **Red** ball wins you `$W` (e.g. $50)
  - Drawing a **Black** ball loses you `$L` (e.g. $20)
  - The drawn ball is *not* replaced.

This tests your intuitive grasp of Expected Value and backward induction.

In [ ]:
import random
import time

def calculate_optimal_ev(R, B, win_val, lose_val, stop_val, memo=None):
    """
    Uses dynamic programming (backward induction) to find the exact Expected Value 
    of playing perfectly from this state.
    """
    if memo is None:
        memo = {}
        
    state = (R, B)
    if state in memo:
        return memo[state]
        
    # Base Cases
    if R == 0 and B == 0:
        return stop_val
        
    if R == 0:
        # Only black balls left. Drawing is guaranteed loss. You will just stop.
        return stop_val
        
    if B == 0:
        # Only red balls left. You will draw all of them.
        return stop_val + (R * win_val)
        
    # Recursive Step: Compare stopping now vs EV of drawing
    total_balls = R + B
    prob_red = R / total_balls
    prob_black = B / total_balls
    
    # EV if we draw, including the EV of the remaining sub-game
    ev_draw = (prob_red * (win_val + calculate_optimal_ev(R-1, B, win_val, lose_val, stop_val, memo))) + \
              (prob_black * (-lose_val + calculate_optimal_ev(R, B-1, win_val, lose_val, stop_val, memo)))
              
    # Optimal decision maximizes value
    optimal_ev = max(stop_val, ev_draw)
    
    memo[state] = optimal_ev
    return optimal_ev


In [ ]:
def play_urn_game():
    # Game Settings
    init_R = random.randint(3, 8)
    init_B = random.randint(3, 8)
    win_val = 50
    lose_val = 20
    stop_val = 0
    
    R = init_R
    B = init_B
    cash = 0
    
    print("=== URN DRAW SIMULATOR ===")
    print(f"Initial Urn: {R} Red (+$50) | {B} Black (-$20)")
    print("Stop Value: Guaranteed $0 added to current cash (cash becomes final).\n")
    
    # Calculate exactly what the true EV of the game is right now
    true_initial_ev = calculate_optimal_ev(R, B, win_val, lose_val, stop_val)
    print("(The bot is calculating the perfect mathematical path behind the scenes...)\n")
    
    while R > 0 or B > 0:
        current_optimal_ev = calculate_optimal_ev(R, B, win_val, lose_val, stop_val)
        
        print(f"[State] Red: {R} | Black: {B} | Current Cash: ${cash}")
        choice = input("Do you want to (D)raw or (S)top? ").strip().upper()
        
        if choice == 'S':
            print(f"\n>>> You stopped. Final Cash: ${cash}")
            print(f">>> Mathematically, the EV of continuing was ${current_optimal_ev:.2f}.")
            if current_optimal_ev > stop_val:
                print("    **Mistake!** You gave up positive EV.")
            else:
                print("    **Perfect Play!** You stopped when the game tilted out of your favor.")
            break
            
        elif choice == 'D':
            # Check if drawing was a mistake
            if current_optimal_ev <= stop_val:
                print("\n[!] WARNING: Drawing here was a mathematically -EV decision.")
                
            # Execute draw
            total = R + B
            draw = random.random()
            if draw < (R/total):
                print(f" -> You drew **RED**! You win ${win_val}\n")
                cash += win_val
                R -= 1
            else:
                print(f" -> You drew **BLACK**! You lose ${lose_val}\n")
                cash -= lose_val
                B -= 1
        else:
            print("Invalid choice. Type D or S.")
            
    if R == 0 and B == 0:
        print(f"\n>>> Urn is empty. Final Cash: ${cash}")
        
    print(f"\nFun Fact: The exact Expected Value of the starting game was ${true_initial_ev:.2f}")


In [ ]:
play_urn_game()